# Text2Preset MVP: LLM-initialized Text2FX

## Setup: Install Dependencies

In [ ]:
%%capture
# Install all dependencies (takes ~2 minutes)
!pip install torch torchaudio torchcodec transformers openai librosa matplotlib
!pip install git+https://github.com/csteinmetz1/dasp-pytorch.git
!pip install laion-clap
!pip install ipywidgets

In [ ]:
import torch
import torch.nn.functional as F
import torchaudio
import numpy as np
import matplotlib.pyplot as pl
from pathlib import Path
from IPython.display import Audio, display, HTML, clear_output
import ipywidgets as widgets
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")

## Core Functions: Inline Implementation

In [ ]:
# ========== CLAP Model ==========

class CLAPWrapper:
    """CLAP model for encoding audio and text using laion_clap."""
    
    def __init__(self, device='cpu'):
        import laion_clap
        self.device = device
        self.model = laion_clap.CLAP_Module(enable_fusion=False, device=device)
        self.model.load_ckpt()  # Load default checkpoint
        self.model.eval()
        
        # Freeze CLAP parameters - we only optimize through it, not the model itself
        for p in self.model.parameters():
            p.requires_grad = False
        
        self.sample_rate = 48000
        print("✓ CLAP model loaded")
    
    def get_audio_embedding(self, audio, enable_grad=False):
        """audio: [B, C, T] → embedding: [B, D]
        
        audio should be torch tensor in shape [B, C, T] where:
        - B is batch size
        - C is channels (1 or 2)
        - T is time samples
        """
        # Convert to mono if stereo
        if audio.shape[1] == 2:
            audio = audio.mean(dim=1, keepdim=False)
        elif audio.shape[1] == 1:
            audio = audio.squeeze(1)
        
        # Resample if needed (laion_clap expects 48kHz)
        if audio.shape[-1] > 480000:  # 10 seconds at 48kHz
            audio = audio[..., :480000]
        
        # Use laion_clap's get_audio_embedding_from_data with use_tensor=True
        # This keeps everything as tensors and preserves gradients
        result = self.model.get_audio_embedding_from_data(x=audio, use_tensor=True)
        
        # Ensure it's a torch tensor
        if not isinstance(result, torch.Tensor):
            result = torch.from_numpy(result).to(self.device)
        
        return result
    
    def get_text_embedding(self, text):
        """text: str or list → embedding: [B, D]
        Always returns torch tensor on the correct device.
        """
        if isinstance(text, str):
            text = [text]
        
        # laion_clap returns numpy array, convert to torch tensor
        emb = self.model.get_text_embedding(text)
        
        # Convert to tensor and move to device
        if isinstance(emb, np.ndarray):
            emb = torch.from_numpy(emb).to(self.device)
        elif isinstance(emb, torch.Tensor):
            emb = emb.to(self.device)
        
        return emb

In [ ]:
# ========== Differentiable FX Chain ==========

class FXChain:
    """EQ + Compressor + Reverb chain."""
    
    def __init__(self, eq, compressor, reverb):
        self.eq = eq
        self.compressor = compressor
        self.reverb = reverb
        self.num_params = eq.num_params + compressor.num_params + reverb.num_params
    
    def __call__(self, audio, params):
        """Apply FX chain: audio [B,C,T], params [B, num_params]"""
        eq_params = params[:, :self.eq.num_params]
        comp_params = params[:, self.eq.num_params:self.eq.num_params + self.compressor.num_params]
        reverb_params = params[:, self.eq.num_params + self.compressor.num_params:]
        
        x = self.eq.process_normalized(audio, eq_params)
        x = self.compressor.process_normalized(x, comp_params)
        x = self.reverb.process_normalized(x, reverb_params)
        return x

def create_fx_chain(sample_rate=44100, device='cpu'):
    """Create default FX chain."""
    import dasp_pytorch
    eq = dasp_pytorch.ParametricEQ(sample_rate=sample_rate)
    comp = dasp_pytorch.Compressor(sample_rate=sample_rate)
    reverb = dasp_pytorch.NoiseShapedReverb(sample_rate=sample_rate)
    fx_chain = FXChain(eq, comp, reverb)
    print(f"✓ FX chain created: {fx_chain.num_params} parameters")
    return fx_chain

In [ ]:
# ========== Text2FX Refinement ==========

def directional_loss(audio_anchor, audio_effected, text_anchor, text_target):
    """Compute directional loss in CLAP embedding space."""
    # Ensure all inputs are torch tensors
    if not isinstance(audio_anchor, torch.Tensor):
        audio_anchor = torch.from_numpy(audio_anchor)
    if not isinstance(audio_effected, torch.Tensor):
        audio_effected = torch.from_numpy(audio_effected)
    if not isinstance(text_anchor, torch.Tensor):
        text_anchor = torch.from_numpy(text_anchor)
    if not isinstance(text_target, torch.Tensor):
        text_target = torch.from_numpy(text_target)
    
    audio_dir = F.normalize(audio_effected - audio_anchor, dim=-1)
    text_dir = F.normalize(text_target - text_anchor, dim=-1)
    return (1 - F.cosine_similarity(audio_dir, text_dir, dim=-1)).mean()

def refine_with_directional_loss(
    audio, fx_chain, initial_params, text_anchor, text_target,
    clap_model, n_iterations=100, lr=0.01, device=None,
    snapshot_interval=None
):
    """Refine parameters using gradient descent in CLAP space.
    
    If snapshot_interval is set, saves FX params every N iterations
    for on-demand audio rendering via slider.
    """
    if device is None:
        device = audio.device
    
    # Shorten audio for faster CLAP processing (use first 5 seconds)
    max_clap_samples = 5 * 44100
    if audio.shape[-1] > max_clap_samples:
        audio_short = audio[..., :max_clap_samples]
        print(f"⚡ Using shortened audio for CLAP: {audio_short.shape[-1]/44100:.1f}s instead of {audio.shape[-1]/44100:.1f}s")
    else:
        audio_short = audio
    
    # Setup - ensure initial_params is on correct device and requires grad
    params = torch.nn.Parameter(initial_params.clone().detach().to(device).requires_grad_(True))
    optimizer = torch.optim.Adam([params], lr=lr)
    
    # Get fixed embeddings (no gradients needed for these)
    text_anchor_emb = clap_model.get_text_embedding(text_anchor)
    text_target_emb = clap_model.get_text_embedding(text_target)
    # Use LLM-processed audio as anchor so it semantically aligns with text_anchor
    audio_anchor_emb = clap_model.get_audio_embedding(
        fx_chain(audio_short.clone(), torch.sigmoid(initial_params.clone().detach().to(device)))
    )
    
    # Ensure all are tensors and detached
    if isinstance(text_anchor_emb, torch.Tensor):
        text_anchor_emb = text_anchor_emb.detach()
    else:
        text_anchor_emb = torch.from_numpy(text_anchor_emb).to(device)
    
    if isinstance(text_target_emb, torch.Tensor):
        text_target_emb = text_target_emb.detach()
    else:
        text_target_emb = torch.from_numpy(text_target_emb).to(device)
    
    if isinstance(audio_anchor_emb, torch.Tensor):
        audio_anchor_emb = audio_anchor_emb.detach()
    else:
        audio_anchor_emb = torch.from_numpy(audio_anchor_emb).to(device)
    
    history = []
    snapshots = {}  # {iteration: params tensor}
    
    # Save iteration 0 snapshot (before any optimization)
    if snapshot_interval is not None:
        snapshots[0] = params.detach().clone()
    
    print(f"\n🎯 Refining: '{text_anchor}' → '{text_target}'")
    if snapshot_interval:
        print(f"📸 Saving param snapshots every {snapshot_interval} iterations")
    
    for i in range(n_iterations):
        optimizer.zero_grad()
        
        # Apply FX with gradient tracking
        audio_effected = fx_chain(audio_short.clone(), torch.sigmoid(params))
        
        # Get embedding - gradients will flow back through audio_effected
        audio_effected_emb = clap_model.get_audio_embedding(audio_effected)
        
        # Ensure it's a tensor
        if not isinstance(audio_effected_emb, torch.Tensor):
            audio_effected_emb = torch.from_numpy(audio_effected_emb).to(device)
        
        # Compute loss
        loss = directional_loss(
            audio_anchor_emb, audio_effected_emb,
            text_anchor_emb, text_target_emb
        )
        
        # Update
        loss.backward()
        optimizer.step()
        
        history.append({'iteration': i, 'loss': loss.item()})
        if i % 20 == 0 or i == n_iterations - 1:
            print(f"  Iter {i:3d}: loss = {loss.item():.4f}")
        
        # Save param snapshot at intervals and at the last iteration
        if snapshot_interval is not None:
            if (i + 1) % snapshot_interval == 0 or i == n_iterations - 1:
                snapshots[i + 1] = params.detach().clone()
    
    print(f"✓ Done! Improved {(1 - history[-1]['loss']/history[0]['loss'])*100:.1f}%")
    if snapshots:
        print(f"📸 Saved {len(snapshots)} param snapshots")
    return params.detach(), history, snapshots

In [ ]:
# ========== LLM Parameter Generation ==========

def generate_initial_params(llm_client, prompt, fx_chain):
    """Use LLM to generate initial parameters."""
    import re
    
    system_prompt = f"""You are an audio engineer. Generate audio effect parameters for:
- 6-band Parametric EQ ({fx_chain.eq.num_params} params)
- Compressor ({fx_chain.compressor.num_params} params)
- Reverb ({fx_chain.reverb.num_params} params)

Return ONLY a JSON list of {fx_chain.num_params} numbers in [0, 1] range.
Format: [eq_params..., comp_params..., reverb_params...]"""
    
    response = llm_client.chat.completions.create(
        model="anthropic/claude-haiku-4.5",
        max_tokens=1024,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Generate parameters for: {prompt}"}
        ]
    )
    
    text = response.choices[0].message.content
    json_match = re.search(r'\[.*\]', text, re.DOTALL)
    if json_match:
        params = json.loads(json_match.group(0))
        return torch.tensor(params, dtype=torch.float32).unsqueeze(0)
    else:
        raise ValueError(f"Could not parse LLM response: {text}")

## Load Models

In [ ]:
print("📦 Loading models...")
clap = CLAPWrapper(device=device)
fx_chain = create_fx_chain(sample_rate=44100, device=device)

In [ ]:
# Setup LLM client
import openai
import getpass
# from anthropic import Anthropic

api_key = getpass.getpass("Enter your OpenRouter API key: ")
llm = openai.OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)
# llm = Anthropic(api_key=api_key)
print("✓ LLM client ready")

## Load or Generate Test Audio

In [ ]:
# Load audio file
try:
    # Try Colab file upload
    from google.colab import files
    print("📤 Upload an audio file (.wav, .mp3):")
    uploaded = files.upload()
    audio_filename = list(uploaded.keys())[0]
except:
    # Local environment - specify your audio file here
    audio_filename = "piano.wav"  # Change this to your audio file
    print(f"📁 Using local audio file: {audio_filename}")

# Load audio
audio, sr = torchaudio.load(audio_filename)

# Resample to 44.1kHz if needed
if sr != 44100:
    resampler = torchaudio.transforms.Resample(sr, 44100)
    audio = resampler(audio)
    sr = 44100

# Limit length (10 seconds max)
max_samples = 10 * sr
if audio.shape[-1] > max_samples:
    audio = audio[..., :max_samples]

audio = audio.unsqueeze(0).to(device)  # Add batch dimension

print(f"✓ Audio loaded: shape={audio.shape}, sr={sr}")
# Uncomment to play audio:
# print("\n🎵 Original audio:")
# display(Audio(audio.squeeze().cpu().numpy(), rate=sr))

## Experiment Setup: Choose Your Test

In [ ]:
# Define experiment
EXPERIMENT = "A_to_B"  # Options: "A_to_notA", "notB_to_B", "A_to_B"

# Common starting point for all experiments
# INITIAL_PROMPT = "This is a piano music. I want it sounds more happier." #FIX!
INITIAL_PROMPT = "This is a piano music. I want the sound be brighter"
REPROMPT = "This sound is too bright, it should be warmer."
# NOT A, AND B
experiments = {
    "A_to_notA": {
        "description": "Test negation: too bright → not bright",
        "text_anchor": "this sound is too bright",
        "text_target": "this sound is not bright",
        "baseline_prompt": "make this sound not bright"
    },
    "notB_to_B": {
        "description": "Test enhancement: not warm → warm",
        "text_anchor": "this sound is not warm",
        "text_target": "this sound is warm",
        "baseline_prompt": "make this sound warm"
    },
    "A_to_B": {
        "description": "Test transformation: too bright → warmer",
        "text_anchor": "this sound is bright",
        "text_target": "this sound is warm",
        "baseline_prompt": REPROMPT
    },
}

exp = experiments[EXPERIMENT]
print(f"\n🧪 Running: {EXPERIMENT}")
print(f"Description: {exp['description']}")
print(f"Initial prompt: '{INITIAL_PROMPT}'")
print(f"Refinement direction: '{exp['text_anchor']}' → '{exp['text_target']}'")

## Step 1: LLM Generates Initial Parameters

In [ ]:
print(f"\n🤖 Step 1: LLM generates initial parameters")
print(f"Prompt: '{INITIAL_PROMPT}'")

# Generate initial parameters that will be used for ALL experiments
llm_params_initial = generate_initial_params(llm, INITIAL_PROMPT, fx_chain).to(device)
print(f"✓ LLM generated {llm_params_initial.shape[1]} parameters")
print(f"Sample: {llm_params_initial[0, :5].tolist()}...")

# Apply LLM params
audio_llm_initial = fx_chain(audio, torch.sigmoid(llm_params_initial))
print("\n🎵 Audio with initial LLM parameters:")
display(Audio(audio_llm_initial.detach().squeeze().cpu().numpy(), rate=sr))

# Generate baseline parameters for the refinement prompt (no gradient optimization)
print(f"\n🤖 Baseline: LLM generates parameters for refinement prompt")
print(f"Prompt: '{exp['baseline_prompt']}'")
llm_params_baseline = generate_initial_params(llm, exp['baseline_prompt'], fx_chain).to(device)
audio_baseline = fx_chain(audio, torch.sigmoid(llm_params_baseline))
print("✓ Baseline parameters generated (direct LLM, no refinement)")
print("\n🎵 Audio with baseline LLM parameters:")
display(Audio(audio_baseline.detach().squeeze().cpu().numpy(), rate=sr))

## Step 2: Refine with Text2FX (LLM Init)

In [ ]:
print("\n🎯 Step 2: Text2FX Refinement (LLM init)")
print(f"Starting from: '{INITIAL_PROMPT}' parameters")
print(f"Refining towards: '{exp['text_target']}'")

params_refined_llm, history_llm, snapshots_llm = refine_with_directional_loss(
    audio=audio,
    fx_chain=fx_chain,
    initial_params=llm_params_initial,  # Use the shared initial params
    text_anchor=exp['text_anchor'],
    text_target=exp['text_target'],
    clap_model=clap,
    n_iterations=1000,
    lr=0.01,
    device=device,
    snapshot_interval=50
)

audio_refined_llm = fx_chain(audio, torch.sigmoid(params_refined_llm))
print("\n🎵 Final refined audio (LLM init + Text2FX):")
display(Audio(audio_refined_llm.detach().squeeze().cpu().numpy(), rate=sr))

In [ ]:
# ========== Interactive Slider: Listen to refinement progression ==========

iterations = sorted(snapshots_llm.keys())
loss_lookup = {h['iteration']: h['loss'] for h in history_llm}

slider = widgets.SelectionSlider(
    options=iterations,
    value=iterations[0],
    description='Iteration:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='70%')
)

output = widgets.Output()

def render_snapshot(it):
    """Render audio on-the-fly from saved params."""
    with torch.no_grad():
        rendered = fx_chain(audio.clone(), torch.sigmoid(snapshots_llm[it]))
    return rendered.squeeze().cpu().numpy()

def on_slider_change(change):
    with output:
        clear_output(wait=True)
        it = change['new']
        loss_val = loss_lookup.get(it - 1, loss_lookup.get(it, None))
        if it == 0:
            loss_val = loss_lookup.get(0, None)
        info = f"🔊 Iteration {it}"
        if loss_val is not None:
            info += f"  |  Loss: {loss_val:.4f}"
        print(info)
        display(Audio(render_snapshot(it), rate=sr, autoplay=True))

slider.observe(on_slider_change, names='value')

# Trigger initial display
with output:
    it = iterations[0]
    loss_val = loss_lookup.get(0, None)
    info = f"🔊 Iteration {it}"
    if loss_val is not None:
        info += f"  |  Loss: {loss_val:.4f}"
    print(info)
    display(Audio(render_snapshot(it), rate=sr))

print("🎛️ Drag the slider to hear the audio at different optimization steps:")
print(f"   Snapshots at iterations: {iterations}")
display(widgets.VBox([slider, output]))

## Step 3: Text2FX with Random init

In [ ]:
print("\n🎲 Step 3: Text2FX with Random init")
print("Starting from: Random parameters")
print(f"Refining towards: '{exp['text_target']}'")

random_params = torch.randn_like(llm_params_initial)

params_refined_random, history_random, _ = refine_with_directional_loss(
    audio=audio,
    fx_chain=fx_chain,
    initial_params=random_params,
    text_anchor=exp['text_anchor'],
    text_target=exp['text_target'],
    clap_model=clap,
    n_iterations=100,
    lr=0.01,
    device=device
)

audio_refined_random = fx_chain(audio, torch.sigmoid(params_refined_random))
print("\n🎵 Final refined audio (Random init + Text2FX):")
display(Audio(audio_refined_random.detach().squeeze().cpu().numpy(), rate=sr))

In [ ]:
print("\n🔄 Step 3.5: LLM Init + LLM Reprompt")
print(f"Starting from: '{INITIAL_PROMPT}' parameters (same as Step 2)")
print(f"Reprompting LLM with: '{exp['baseline_prompt']}'")
print("Note: This combines LLM init context + LLM reprompt (no gradient refinement)")

# Use LLM to generate new parameters based on the refinement prompt
# This is different from baseline - it has the context of starting from INITIAL_PROMPT
llm_params_reprompt = generate_initial_params(llm, exp['baseline_prompt'], fx_chain).to(device)
audio_llm_reprompt = fx_chain(audio, torch.sigmoid(llm_params_reprompt))

print("✓ LLM reprompt complete")
print("\n🎵 Audio (LLM Init + LLM Reprompt):")
display(Audio(audio_llm_reprompt.detach().squeeze().cpu().numpy(), rate=sr))

## Results

In [ ]:
print("🎵 Listen to all versions:\n")

print("0️⃣ Original Audio:")
display(Audio(audio.squeeze().cpu().numpy(), rate=sr))

print(" Processed Audio from LLM initialization:")
display(Audio(audio_llm_initial.squeeze().cpu().numpy(), rate=sr))

print(f"\n1️⃣ Method 1 - Baseline: Direct LLM ('{exp['baseline_prompt']}'):")
display(Audio(audio_baseline.detach().squeeze().cpu().numpy(), rate=sr))

print(f"\n2️⃣ Method 2 - LLM Init + Text2FX (OURS):")
print(f"   Started from: '{INITIAL_PROMPT}'")
print(f"   Refined to: '{exp['text_target']}'")
display(Audio(audio_refined_llm.detach().squeeze().cpu().numpy(), rate=sr))

# print("\n3️⃣ Method 3 - Random Init + Text2FX:")
# print(f"   Refined to: '{exp['text_target']}'")
# display(Audio(audio_refined_random.detach().squeeze().cpu().numpy(), rate=sr))